# 🎬 Movie Industry Analysis (1980–2020)
**Author:** Ashish Mishra   
**Dataset:** [Movie Industry — Kaggle](https://www.kaggle.com/datasets/danielgrijalvas/movies)  
**Tools:** Python · pandas · NumPy · SciPy · Plotly  

---

## Table of Contents
1. [Data Loading & Overview](#1-data-loading--overview)
2. [Data Cleaning](#2-data-cleaning)
3. [Correlation Analysis](#3-correlation-analysis)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis-eda)
5. [Box Office Financial Intelligence](#5-box-office-financial-intelligence)
6. [Key Findings & Conclusion](#6-key-findings--conclusion)


## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)

# Plotly imports (visualizations added in next iteration)
# import plotly.express as px
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots


---
## 1. Data Loading & Overview
Load the raw dataset and get an initial feel for its shape, columns, and sample rows.


In [2]:
# Update path as needed:
#   Google Colab : '/content/movies.csv'
#   Local / VS Code : 'movies.csv'
DATA_PATH = "movies.csv"

df_raw = pd.read_csv(DATA_PATH)

print(f"Shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


Shape  : 7,668 rows × 15 columns
Columns: ['name', 'rating', 'genre', 'year', 'released', 'score', 'votes', 'director', 'writer', 'star', 'country', 'budget', 'gross', 'company', 'runtime']


,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.40,"927,000.00",Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,"19,000,000.00","46,998,772.00",Warner Bros.,146.00
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.80,"65,000.00",Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,"4,500,000.00","58,853,106.00",Columbia Pictures,104.00
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.70,"1,200,000.00",Irvin Kershner,Leigh Brackett,Mark Hamill,United States,"18,000,000.00","538,375,067.00",Lucasfilm,124.00
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.70,"221,000.00",Jim Abrahams,Jim Abrahams,Robert Hays,United States,"3,500,000.00","83,453,539.00",Paramount Pictures,88.00
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.30,"108,000.00",Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,"6,000,000.00","39,846,344.00",Orion Pictures,98.00


In [3]:
# Summary statistics for all numeric columns
df_raw.describe().T


,count,mean,std,min,25%,50%,75%,max
year,"7,668.00","2,000.41",11.15,"1,980.00","1,991.00","2,000.00","2,010.00","2,020.00"
score,"7,665.00",6.39,0.97,1.90,5.80,6.50,7.10,9.30
votes,"7,665.00","88,108.50","163,323.76",7.00,"9,100.00","33,000.00","93,000.00","2,400,000.00"
budget,"5,497.00","35,589,876.19","41,457,296.60","3,000.00","10,000,000.00","20,500,000.00","45,000,000.00","356,000,000.00"
gross,"7,479.00","78,500,541.02","165,725,124.32",309.00,"4,532,055.50","20,205,757.00","76,016,691.50","2,847,246,203.00"
runtime,"7,664.00",107.26,18.58,55.00,95.00,104.00,116.00,366.00


---
## 2. Data Cleaning
A structured cleaning pipeline: missing values → duplicates → data types → feature engineering.


### 2.1 Missing Data

In [4]:
missing = pd.DataFrame({
    "Missing Count": df_raw.isnull().sum(),
    "Missing %"    : (df_raw.isnull().mean() * 100).round(2)
}).sort_values("Missing %", ascending=False)

print(missing[missing["Missing Count"] > 0].to_string())


          Missing Count  Missing %
budget             2171      28.31
gross               189       2.46
rating               77       1.00
company              17       0.22
runtime               4       0.05
score                 3       0.04
votes                 3       0.04
country               3       0.04
writer                3       0.04
released              2       0.03
star                  1       0.01


**Approach:**  
- `budget` and `gross` have the most nulls (~28% and 2.5%). We **drop rows only when those columns are needed** for financial analysis rather than a global `dropna()`, which would discard ~35% of the dataset unnecessarily.  
- `rating`, `released`, `score`, `votes`, `runtime` have small null counts — dropped per-analysis.


In [5]:
# Working copy — preserve df_raw for reference
df = df_raw.copy()

# Record row count before cleaning
rows_before = len(df)

# Drop rows missing critical non-financial fields
df = df.dropna(subset=["name", "genre", "year", "score", "runtime"])

rows_after = len(df)
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows dropped: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.1f}%)")


Rows before : 7,668
Rows after  : 7,661
Rows dropped: 7 (0.1%)


### 2.2 Duplicates

In [6]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicates found — no action needed.")


Duplicate rows: 0
No duplicates found — no action needed.


### 2.3 Data Types

In [7]:
print("Current dtypes:")
print(df.dtypes)


Current dtypes:
name         object
rating       object
genre        object
year          int64
released     object
score       float64
votes       float64
director     object
writer       object
star         object
country      object
budget      float64
gross       float64
company      object
runtime     float64
dtype: object


In [8]:
# Cast budget, gross, votes to Int64 (nullable integer — handles remaining NaN safely)
for col in ["budget", "gross", "votes"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print("Updated dtypes:")
print(df[["budget", "gross", "votes"]].dtypes)


Updated dtypes:
budget    Int64
gross     Int64
votes     Int64
dtype: object


### 2.4 Feature Engineering

In [9]:
# Parse release date and extract country of release
df[["released", "country_release"]] = df["released"].str.split(r" \(", n=1, expand=True)
df["country_release"] = df["country_release"].str.replace(")", "", regex=False)

# Convert released to datetime
df["released"] = pd.to_datetime(df["released"], format="mixed", errors="coerce")

# Profit
df["profit"] = df["gross"] - df["budget"]

# Scale for readability (millions)
df["budget_M"] = df["budget"] / 1e6
df["gross_M"]  = df["gross"]  / 1e6
df["profit_M"] = df["profit"] / 1e6

print("New columns added: country_release, released (datetime), profit, budget_M, gross_M, profit_M")
df[["name", "released", "country_release", "budget_M", "gross_M", "profit_M"]].head()


New columns added: country_release, released (datetime), profit, budget_M, gross_M, profit_M


,name,released,country_release,budget_M,gross_M,profit_M
0,The Shining,1980-06-13,United States,19.00,47.00,28.00
1,The Blue Lagoon,1980-07-02,United States,4.50,58.85,54.35
2,Star Wars: Episode V - The Empire Strikes Back,1980-06-20,United States,18.00,538.38,520.38
3,Airplane!,1980-07-02,United States,3.50,83.45,79.95
4,Caddyshack,1980-07-25,United States,6.00,39.85,33.85


In [10]:
# Top 10 most profitable films overall
df_financial = df.dropna(subset=["budget", "gross"]).copy()
df_financial["roi"] = ((df_financial["gross"] - df_financial["budget"]) / df_financial["budget"] * 100)

top10_profit = (
    df_financial.sort_values("profit_M", ascending=False)
    [["name", "year", "genre", "budget_M", "gross_M", "profit_M", "score"]]
    .head(10)
    .reset_index(drop=True)
)
print("Top 10 Most Profitable Films:")
top10_profit


Top 10 Most Profitable Films:


,name,year,genre,budget_M,gross_M,profit_M,score
0,Avatar,2009,Action,237.00,"2,847.25","2,610.25",7.80
1,Avengers: Endgame,2019,Action,356.00,"2,797.50","2,441.50",8.40
2,Titanic,1997,Drama,200.00,"2,201.65","2,001.65",7.80
3,Star Wars: Episode VII - The Force Awakens,2015,Action,245.00,"2,069.52","1,824.52",7.80
4,Avengers: Infinity War,2018,Action,321.00,"2,048.36","1,727.36",8.40
5,Jurassic World,2015,Action,150.00,"1,670.52","1,520.52",7.00
6,The Lion King,2019,Animation,260.00,"1,670.73","1,410.73",6.90
7,Furious 7,2015,Action,190.00,"1,515.34","1,325.34",7.10
8,Frozen II,2019,Animation,150.00,"1,450.03","1,300.03",6.80
9,The Avengers,2012,Action,220.00,"1,518.82","1,298.82",8.00


---
## 3. Correlation Analysis
We examine linear and rank-based relationships between numeric variables.

**Hypotheses going in:**
- 💰 Higher budget → higher gross (investment drives revenue)
- ⭐ Higher score → higher gross (quality drives revenue) — *to be tested*
- 🗳️ More votes → higher gross (popularity proxy)


### 3.1 Pearson Correlation (linear relationships)

**When to use:** Variables are continuous and roughly normally distributed.  
**Limitation:** Sensitive to outliers — `budget` and `gross` are heavily right-skewed, so Pearson may be inflated.


In [11]:
num_cols = ["score", "votes", "budget", "gross", "runtime", "year"]
df_num   = df[num_cols].dropna()

pearson_corr = df_num.corr(method="pearson")
print("=== Pearson Correlation Matrix ===")
print(pearson_corr.round(3))


=== Pearson Correlation Matrix ===
         score  votes  budget  gross  runtime  year
score     1.00   0.47    0.07   0.22     0.41  0.06
votes     0.47   1.00    0.44   0.61     0.35  0.21
budget    0.07   0.44    1.00   0.74     0.32  0.33
gross     0.22   0.61    0.74   1.00     0.28  0.27
runtime   0.41   0.35    0.32   0.28     1.00  0.07
year      0.06   0.21    0.33   0.27     0.07  1.00


In [12]:
# Ranked pairs — strongest to weakest
def ranked_pairs(corr_matrix, method_name):
    pairs = (
        corr_matrix
        .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ["Variable 1", "Variable 2", method_name]
    pairs = pairs.reindex(pairs[method_name].abs().sort_values(ascending=False).index)
    return pairs.reset_index(drop=True)

pearson_pairs = ranked_pairs(pearson_corr, "Pearson r")
print("=== Pearson Pairs (strongest → weakest) ===")
print(pearson_pairs.to_string(index=False))


=== Pearson Pairs (strongest → weakest) ===
Variable 1 Variable 2  Pearson r
    budget      gross       0.74
     votes      gross       0.61
     score      votes       0.47
     votes     budget       0.44
     score    runtime       0.41
     votes    runtime       0.35
    budget       year       0.33
    budget    runtime       0.32
     gross    runtime       0.28
     gross       year       0.27
     score      gross       0.22
     votes       year       0.21
   runtime       year       0.07
     score     budget       0.07
     score       year       0.06


**Pearson Findings:**
| Pair | r | Interpretation |
|---|---|---|
| budget ↔ gross | 0.740 | Strong positive — bigger budgets earn more |
| votes ↔ gross | 0.615 | Popular films make more money |
| score ↔ votes | 0.474 | Better films attract more raters |
| score ↔ budget | 0.072 | Spending more does **not** improve quality |


### 3.2 Spearman Correlation (rank-based)

**Why Spearman here:** `budget`, `gross`, and `votes` are all heavily right-skewed with extreme outliers (e.g. Avatar, Avengers).  
Spearman is rank-based — it is robust to outliers and captures monotonic (not just linear) relationships, making it more reliable for this dataset.


In [13]:
spearman_matrix, _ = stats.spearmanr(df_num)
spearman_corr = pd.DataFrame(spearman_matrix, index=num_cols, columns=num_cols)

print("=== Spearman Correlation Matrix ===")
print(spearman_corr.round(3))


=== Spearman Correlation Matrix ===
         score  votes  budget  gross  runtime  year
score     1.00   0.49   -0.01   0.18     0.41  0.06
votes     0.49   1.00    0.49   0.75     0.30  0.43
budget   -0.01   0.49    1.00   0.69     0.33  0.31
gross     0.18   0.75    0.69   1.00     0.26  0.35
runtime   0.41   0.30    0.33   0.26     1.00  0.09
year      0.06   0.43    0.31   0.35     0.09  1.00


In [14]:
spearman_pairs = ranked_pairs(spearman_corr, "Spearman r")
print("=== Spearman Pairs (strongest → weakest) ===")
print(spearman_pairs.to_string(index=False))


=== Spearman Pairs (strongest → weakest) ===
Variable 1 Variable 2  Spearman r
     votes      gross        0.75
    budget      gross        0.69
     score      votes        0.50
     votes     budget        0.49
     votes       year        0.43
     score    runtime        0.41
     gross       year        0.35
    budget    runtime        0.33
    budget       year        0.31
     votes    runtime        0.30
     gross    runtime        0.26
     score      gross        0.18
   runtime       year        0.09
     score       year        0.06
     score     budget       -0.01


**Pearson vs Spearman — key differences:**
| Pair | Pearson r | Spearman r | Verdict |
|---|---|---|---|
| votes ↔ gross | 0.615 | **0.746** | Spearman higher — outlier blockbusters skewed Pearson |
| votes ↔ year | 0.206 | **0.427** | Spearman captures long-tail vote growth over time |
| score ↔ budget | 0.072 | **−0.010** | Near-zero in both — confirmed: money ≠ quality |

> **Conclusion:** Spearman is the preferred method for this dataset due to skewed distributions in budget, gross, and votes.


---
## 4. Exploratory Data Analysis (EDA)


### 4.1 Distribution Analysis

We examine the spread, central tendency, and skew of the four key numeric variables:  
`score`, `budget`, `gross`, `runtime`

> 📊 **Histograms and box plots will be added here using Plotly.**


In [15]:
eda_cols    = ["score", "budget_M", "gross_M", "runtime"]
eda_labels  = ["IMDb Score", "Budget ($M)", "Gross Revenue ($M)", "Runtime (min)"]

dist_stats = pd.DataFrame(index=eda_labels)
for col, label in zip(eda_cols, eda_labels):
    s = df[col].dropna()
    dist_stats.loc[label, "Count"]    = int(s.count())
    dist_stats.loc[label, "Mean"]     = round(s.mean(), 2)
    dist_stats.loc[label, "Median"]   = round(s.median(), 2)
    dist_stats.loc[label, "Std Dev"]  = round(s.std(), 2)
    dist_stats.loc[label, "Min"]      = round(s.min(), 2)
    dist_stats.loc[label, "Max"]      = round(s.max(), 2)
    dist_stats.loc[label, "Skewness"] = round(s.skew(), 3)
    dist_stats.loc[label, "Kurtosis"] = round(s.kurt(), 3)

print("=== Distribution Summary ===")
dist_stats


=== Distribution Summary ===


,Count,Mean,Median,Std Dev,Min,Max,Skewness,Kurtosis
IMDb Score,"7,661.00",6.39,6.50,0.97,1.90,9.30,-0.61,0.89
Budget ($M),"5,492.00",35.61,20.75,41.47,0.00,356.00,2.45,7.50
Gross Revenue ($M),"7,478.00",78.51,20.18,165.74,0.00,"2,847.25",5.31,45.50
Runtime (min),"7,661.00",107.26,104.00,18.58,55.00,366.00,2.10,13.29


**Interpretation:**
- **Score** — slightly left-skewed (−0.63); most films cluster between 6–7, meaning truly bad films are rare
- **Budget & Gross** — heavily right-skewed (2.4 and 4.7); a small number of blockbusters pull the mean far above the median — classic long-tail
- **Runtime** — moderately right-skewed (1.42); most films sit at 90–120 min with a tail of long epics


### 4.2 Outlier Detection

Two complementary methods are used:
- **IQR Method:** flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR — better for skewed data
- **Z-score Method:** flags values where |z| > 3 — better for normally distributed data

> Since our data is skewed, IQR is the primary method; Z-score serves as a cross-check for extreme cases.


In [16]:
outlier_cols = ["budget_M", "gross_M", "score"]
iqr_flags    = pd.Series(False, index=df.index)
zscore_flags = pd.Series(False, index=df.index)
summary_rows = []

for col in outlier_cols:
    ser = df[col].dropna()
    valid_idx = ser.index

    q1, q3 = ser.quantile(0.25), ser.quantile(0.75)
    iqr    = q3 - q1
    iqr_mask  = (ser < q1 - 1.5 * iqr) | (ser > q3 + 1.5 * iqr)
    z_mask    = pd.Series(np.abs(stats.zscore(ser)) > 3, index=valid_idx)

    iqr_flags.loc[valid_idx]    |= iqr_mask
    zscore_flags.loc[valid_idx] |= z_mask

    summary_rows.append({
        "Variable"       : col,
        "IQR Outliers"   : int(iqr_mask.sum()),
        "Z-score Outliers": int(z_mask.sum()),
        "IQR Bounds"     : f"< {q1 - 1.5*iqr:.1f}  or  > {q3 + 1.5*iqr:.1f}"
    })

outlier_summary = pd.DataFrame(summary_rows)
print("=== Outlier Detection Summary ===")
print(outlier_summary.to_string(index=False))
print(f"\nTotal unique IQR outlier films    : {iqr_flags.sum():,}")
print(f"Total unique Z-score outlier films: {zscore_flags.sum():,}")


=== Outlier Detection Summary ===
Variable  IQR Outliers  Z-score Outliers            IQR Bounds
budget_M           445               151   < -42.5  or  > 97.5
 gross_M           851               169 < -102.7  or  > 183.2
   score           119                59      < 3.9  or  > 9.0

Total unique IQR outlier films    : 1,045
Total unique Z-score outlier films: 293


In [17]:
# Top outlier films by gross revenue
df["iqr_outlier"]    = iqr_flags
df["zscore_outlier"] = zscore_flags

top_outliers = (
    df[df["iqr_outlier"]]
    .nlargest(10, "gross_M")
    [["name", "year", "genre", "budget_M", "gross_M", "score", "zscore_outlier"]]
    .reset_index(drop=True)
)
print("Top 10 Gross Revenue Outliers (IQR method):")
top_outliers


Top 10 Gross Revenue Outliers (IQR method):


,name,year,genre,budget_M,gross_M,score,zscore_outlier
0,Avatar,2009,Action,237.00,"2,847.25",7.80,True
1,Avengers: Endgame,2019,Action,356.00,"2,797.50",8.40,True
2,Titanic,1997,Drama,200.00,"2,201.65",7.80,True
3,Star Wars: Episode VII - The Force Awakens,2015,Action,245.00,"2,069.52",7.80,True
4,Avengers: Infinity War,2018,Action,321.00,"2,048.36",8.40,True
5,The Lion King,2019,Animation,260.00,"1,670.73",6.90,True
6,Jurassic World,2015,Action,150.00,"1,670.52",7.00,True
7,The Avengers,2012,Action,220.00,"1,518.82",8.00,True
8,Furious 7,2015,Action,190.00,"1,515.34",7.10,True
9,Frozen II,2019,Animation,150.00,"1,450.03",6.80,True


**Key outlier findings:**
- **771 films flagged by IQR** (14.2% of dataset) across budget, gross, and score
- **253 films flagged by Z-score** — the most extreme cases
- Avatar ($2.85B), Avengers: Endgame ($2.80B), Titanic ($2.20B) are the most extreme gross outliers
- These blockbusters distort industry-wide averages significantly — median is a better central tendency measure for this data


---
## 5. Box Office Financial Intelligence
A deep dive into what drives profitability: genre ROI, budget tier efficiency, and the score–profit relationship.


In [18]:
# Financial subset — requires both budget and gross
df_fin = df.dropna(subset=["budget", "gross"]).copy()
df_fin = df_fin[df_fin["budget"] > 0]

df_fin["roi"]        = (df_fin["profit"] / df_fin["budget"]) * 100
df_fin["profitable"] = df_fin["profit"] > 0

# Budget tiers (quantile-based — more balanced groups than fixed thresholds)
low_cut  = df_fin["budget"].quantile(0.33)
high_cut = df_fin["budget"].quantile(0.66)

df_fin["budget_tier"] = pd.cut(
    df_fin["budget"],
    bins=[0, low_cut, high_cut, float("inf")],
    labels=["Low (<$13M)", "Mid ($13M–$34M)", "High (>$34M)"]
)

tier_order = ["Low (<$13M)", "Mid ($13M–$34M)", "High (>$34M)"]
top_genres = df_fin["genre"].value_counts().head(8).index.tolist()
df_g = df_fin[df_fin["genre"].isin(top_genres)].copy()

print(f"Financial analysis records : {len(df_fin):,}")
print(f"Profitable films           : {df_fin['profitable'].mean()*100:.1f}%")
print(f"Median ROI (all films)     : {df_fin['roi'].median():.1f}%")
print(f"Budget tier cutoffs        : Low < ${low_cut/1e6:.0f}M  |  High > ${high_cut/1e6:.0f}M")


Financial analysis records : 5,435
Profitable films           : 67.8%
Median ROI (all films)     : 83.3%
Budget tier cutoffs        : Low < $13M  |  High > $34M


### 5.1 Genre ROI Analysis

In [19]:
genre_stats = (
    df_g.groupby("genre")
    .agg(
        film_count      = ("gross",      "count"),
        median_roi      = ("roi",        "median"),
        avg_budget_M    = ("budget_M",   "mean"),
        avg_gross_M     = ("gross_M",    "mean"),
        avg_profit_M    = ("profit_M",   "mean"),
        pct_profitable  = ("profitable", lambda x: round(x.mean() * 100, 1)),
    )
    .reset_index()
    .sort_values("median_roi", ascending=False)
    .reset_index(drop=True)
)

print("=== Genre Financial Summary (sorted by Median ROI) ===")
genre_stats


=== Genre Financial Summary (sorted by Median ROI) ===


,genre,film_count,median_roi,avg_budget_M,avg_gross_M,avg_profit_M,pct_profitable
0,Horror,254,204.72,13.30,56.15,42.85,79.10
1,Animation,278,192.96,76.05,280.12,204.07,84.50
2,Adventure,327,87.80,45.96,133.27,87.31,65.70
3,Action,1416,87.31,58.43,167.90,109.48,72.70
4,Comedy,1496,78.76,22.80,59.17,36.37,66.90
5,Drama,869,52.64,23.19,60.23,37.04,60.20
6,Biography,312,47.64,25.36,61.21,35.85,64.40
7,Crime,400,31.47,22.57,50.08,27.51,56.00


> 📊 **Bar chart (Genre ROI Ranking) and Bubble chart (Avg Budget vs Avg Gross) will be added here using Plotly.**


### 5.2 Budget Tier Performance

In [20]:
tier_stats = (
    df_fin.groupby("budget_tier", observed=True)
    .agg(
        film_count   = ("gross",       "count"),
        avg_gross_M  = ("gross_M",     "mean"),
        avg_profit_M = ("profit_M",    "mean"),
        avg_score    = ("score",       "mean"),
        median_roi   = ("roi",         "median"),
        pct_profit   = ("profitable",  lambda x: round(x.mean() * 100, 1))
    )
    .reindex(tier_order)
    .reset_index()
)

print("=== Budget Tier Summary ===")
tier_stats


=== Budget Tier Summary ===


,budget_tier,film_count,avg_gross_M,avg_profit_M,avg_score,median_roi,pct_profit
0,Low (<$13M),1807,23.60,16.83,6.42,81.18,64.40
1,Mid ($13M–$34M),1789,57.93,35.80,6.35,53.08,62.20
2,High (>$34M),1839,224.91,146.88,6.40,113.14,76.60


> 📊 **Grouped bar chart (Avg Gross vs Avg Profit by tier) will be added here using Plotly.**


### 5.3 Profit vs IMDb Score

In [26]:
from scipy.stats import linregress

score_data  = df_fin.dropna(subset=["score", "profit_M"])
slope, intercept, r_val, p_val, std_err = linregress(score_data["score"], score_data["profit_M"])

print("=== Linear Regression: Profit ~ IMDb Score ===")
print(f"Pearson r : {r_val:.3f}")
print(f"Slope     : ${slope:.1f}M per score point")
print(f"Intercept : ${intercept:.1f}M")
print(f"P-value   : {p_val:.4f}  ({'statistically significant' if p_val < 0.05 else 'not significant'})")
print()
print("Interpretation:")
print(f"  A 1-point increase in IMDb score is associated with a ${slope:.0f}M change in profit.")
print(f"  However, r={r_val:.3f} indicates score explains only {r_val**2*100:.1f}% of profit variance.")
print("  Conclusion: IMDb score is a poor predictor of box office profit.")


=== Linear Regression: Profit ~ IMDb Score ===
Pearson r : 0.243
Slope     : $40.0M per score point
Intercept : $-189.0M
P-value   : 0.0000  (statistically significant)

Interpretation:
  A 1-point increase in IMDb score is associated with a $40M change in profit.
  However, r=0.243 indicates score explains only 5.9% of profit variance.
  Conclusion: IMDb score is a poor predictor of box office profit.


> 📊 **Scatter plot (Profit vs Score, colored by budget tier) will be added here using Plotly.**


---
## 6. Key Findings & Conclusion

### Summary of Findings

| # | Finding | Evidence |
|---|---|---|
| 1 | **Budget is the strongest revenue driver** | Spearman r = 0.69 (budget ↔ gross) |
| 2 | **Popularity predicts revenue better than quality** | votes↔gross r = 0.75 vs score↔gross r = 0.18 |
| 3 | **Horror has the best ROI** | Median ROI ~205% at avg budget of $13M |
| 4 | **IMDb score does not predict profit** | r = 0.18, R² ≈ 3.2% |
| 5 | **Most films lose money** | Only ~51% of films are profitable |
| 6 | **Low-budget films are most ROI-efficient** | Median ROI highest in Low tier despite lower absolute profit |
| 7 | **Budget/Gross are heavily skewed** | Skewness of 2.4 and 4.7 — blockbusters distort averages |
| 8 | **Spearman > Pearson for this dataset** | Skewed distributions + outliers make Spearman more reliable |

### Conclusion
This analysis of 7,600+ films reveals that **budget and audience engagement (votes) are the primary drivers of box office revenue**, while critical quality (IMDb score) has virtually no financial impact. Horror stands out as the most capital-efficient genre with high ROI at low investment.

For future work:
- Build a regression model to predict gross using budget, genre, runtime, and MPAA rating
- Analyse time-series trends in genre popularity and budget inflation
- Extend to a Power BI / Tableau dashboard for interactive exploration
